<a href="https://colab.research.google.com/github/cras-lab/OpenAPI/blob/main/%EA%B8%88%EA%B0%90%EC%9B%90_BankEmployees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈을 설치한다.

In [1]:
import requests
import pandas as pd

API 키를 입력 받는다.

In [ ]:
import getpass
API_KEY = getpass.getpass("API KEY: ")

필요한 매개변수를 설정한다.<BR>
이 매개변수 값 일부는 SPEC을 보고 직접 알아내야 한다.

In [3]:
BASE_MM = "202512"      # 기준년월 예: 2025년 12월
TERM = "Q"              # Q: 분기, Y: 연도, H: 반기

LIST_NO = "SA001"      # 은행 > 일반현황 > 임직원현황
ACCOUNT_CD = "A"       # 총임직원

국내 은행 목록을 읽어 올 명령어와 매개변수를 설정한다.

In [4]:
company_url = "http://fisis.fss.or.kr/openapi/companySearch.json"

company_params = {
    "lang": "kr",
    "auth": API_KEY,
    "partDiv": "A"      # A = 국내은행
}

URL과 매개변수를 사용해 실제 값을 얻어온다.

In [5]:
res = requests.get(company_url, params=company_params)
company_data = res.json()

읽어 온 내용을 출력해 본다.

In [ ]:
import json
print(json.dumps(company_data, indent=2, ensure_ascii=False)) # ensure_ascii=False를 해야 한글이 출력된다.

result > list의 목록만 뽑아낸다

In [7]:
banks = pd.DataFrame(company_data["result"]["list"])

결과를 표형식으로 출력해 보자.

In [ ]:
display(banks[["finance_cd", "finance_nm"]])

 국내은행 목록 중 이름에 '[폐]'가 들어간 폐쇄/폐지 은행만 제외하고 새로 생성

In [9]:
target_banks = banks[
    ~banks["finance_nm"].astype(str).str.contains("[폐]", regex=False, na=False)
].copy()

은행이름을 가나다 순으로 소팅해서 정리해서 최종 목록형성

In [10]:
target_banks = target_banks.sort_values("finance_nm").reset_index(drop=True)

최종 대상을 출력해 본다.

In [ ]:
display(target_banks[["finance_cd", "finance_nm"]])

이제 목록에 있는 각 은행에 대해 실제로 임직원 수를 불러온다.

In [13]:
rows = []

for _, bank in target_banks.iterrows():
    finance_cd = bank["finance_cd"]

    stat_url = "http://fisis.fss.or.kr/openapi/statisticsInfoSearch.json"

    stat_params = {
        "lang": "kr",
        "auth": API_KEY,
        "financeCd": finance_cd,
        "listNo": LIST_NO,
        "accountCd": ACCOUNT_CD,
        "term": TERM,
        "startBaseMm": BASE_MM,
        "endBaseMm": BASE_MM
    }

    res = requests.get(stat_url, params=stat_params)
    stat_data = res.json()

    item = stat_data["result"]["list"]

    if isinstance(item, dict):
        item = [item]

    rows.extend(item)

In [ ]:
print( json.dumps(rows, indent=2, ensure_ascii=False ) )

읽어온 목록을 데이터프레임으로 만든다.

In [19]:
raw = pd.DataFrame(rows)

출력해 보자.

In [ ]:
display(raw)

다른 컬럼은 버리고, 은행과 임직원수만 추출한다.

In [21]:
value_col = raw.columns[-1]

result = raw[["finance_nm", value_col]].copy()
result.columns = [ "은행명", "전체임직원수"]

result["전체임직원수"] = (
    result["전체임직원수"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

전체임직원수 필드를 숫자로 바꾼다.

In [22]:
result["전체임직원수"] = pd.to_numeric(result["전체임직원수"], errors="coerce")

전체 표를 "전체임직원수" 열에 따라 정렬한다.

In [23]:
result = result.sort_values("전체임직원수", ascending=False)

최종 결과를 출력한다.

In [ ]:
print(f"\n국내은행별 {BASE_MM} 전체 임직원 수")
display(result)